# Feature Engineering

In [ ]:
import torch
import pyarrow
import timm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
initial_df = pd.read_parquet("../data/processed/anime_data_1.parquet")
current_df = pd.read_parquet("../data/processed/real_data_1.parquet")
df = pd.concat([initial_df, current_df], axis=0)
df['title'].tail(72)

In [ ]:
df.info()
n_rows = len(df)

## Cohort

Combine Season + Year into one "Cohort" variable

In [ ]:
df['cohort'] = df['season'] +  " " + df['year'].astype(int).astype(str)

In [ ]:
df2 = df.drop(columns=['year', 'season'])

## Metrics against Cohort

The plan will be the following:
* score will be z-scored against their cohort
* wc, favorites, and forum are heavily skewed, so we will perform a log1p first before z-scoring
* dropped will turn into (dropped / wc) as "drop rate" and then z-scored
All the z-scoring is done to even out nostalgia biases

In [ ]:
df2['wc'].describe()

In [ ]:
df2['drop_rate'] = df2['dropped'] / df2['wc']
df2['wc'] = np.log1p(df2['wc'])
df2['favorites'] = np.log1p(df2['favorites'])
df2['forum'] = np.log1p(df2['forum'])
df2['dropped'] = np.log1p(df2['dropped'])


In [ ]:
df2['wc_z'] = df2.groupby('cohort')['wc'].transform(lambda x: (x - x.mean()) / x.std())

Repeat for the rest of the metrics:

In [ ]:
df2['forum_z'] = df2.groupby('cohort')['forum'].transform(lambda x: (x - x.mean()) / x.std())

In [ ]:
df2['favorites_z'] = df2.groupby('cohort')['favorites'].transform(lambda x: (x - x.mean()) / x.std())

In [ ]:
df2['score_z'] = df2.groupby('cohort')['score'].transform(lambda x: (x - x.mean()) / x.std())

In [ ]:
df2['dropped_z'] = df2.groupby('cohort')['dropped'].transform(lambda x: (x - x.mean()) / x.std())
df2['drop_rate_z'] = df2.groupby('cohort')['drop_rate'].transform(lambda x: (x - x.mean()) / x.std())

In [ ]:
df3 = df2.drop(columns=['wc', 'score', 'favorites', 'dropped', 'forum'])

As you can notice, there are z-scores that don't have a value. This is because they're the only anime in their cohort. We will replace these with zeroes.

In [ ]:
df4 = df3.fillna({'wc_z' : 0, 'forum_z': 0, 'favorites_z': 0, 'score_z': 0, 'dropped_z': 0, 'drop_rate_z': 0})
df4.info()

In [ ]:
corr_df = df4[['score_z', 'wc_z', 'favorites_z', 'dropped_z', 'forum_z']]
corr_matrix = corr_df.corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', xticklabels=corr_df.columns, yticklabels=corr_df.columns)

## Images

We will soon be using wd-eva02, which needs input resolutions of 448x448. The plan is to pad them so that they become a square, and then resize it down to 448x448.

In [ ]:
def process_image(mal_id):
    image_path =  "../data/images/" + str(mal_id) + ".jpg"
    target_dir = Path("../data/images")
    file_name = str(mal_id) + ".jpg"
    image_path_true = Path(image_path)
    if image_path_true.exists():
        img = Image.open(image_path)
    
        width, height = img.size
        max_side = max(width, height)
        
        new_img = Image.new("RGB", (max_side, max_side), (0, 0, 0))
        left = (max_side - width) // 2
        top = (max_side - height) // 2
        new_img.paste(img, (left, top))
    
        resized_img = new_img.resize((448, 448))
    
        resized_img.save(target_dir / file_name)

Now we iterate throughout the entire processed CSV:

In [ ]:
# display_count = 0

# for mal_id in df4["mal_id"]:
#     process_image(mal_id)
#     if display_count == 200:
#         display_count = 1
#         print(f"Processed 200 images! Current ID: {mal_id}")
#     else:
#         display_count += 1


Now we want to translate these images into vectors that we can actually feed into our neural network. For this, we will use wd-eva02. We will start with the Cowboy Bebop test image.

In [ ]:
# model = timm.create_model("hf_hub:SmilingWolf/wd-eva02-large-tagger-v3", pretrained=True)
# model.eval()

In [ ]:
# data_config = timm.data.resolve_model_data_config(model)
# transforms = timm.data.create_transform(**data_config, is_training=False)
# image_path = "../data/images/1.jpg"
# test_img = Image.open(image_path)
# test_tensor = transforms(test_img).unsqueeze(0)
# with torch.no_grad():
#     embedding = model(test_tensor)

# print("Embedding Shape:", embedding.shape)

In [ ]:
# embedding

Looks good. Let's get all the images from our current mal_id's. If the mal_id doesn't exist in the image folder, we will put a black image by default.

In [ ]:
# def eva_tag(mal_ids):
#     count = 0
#     image_embeds = []
#     for mal_id in mal_ids:
#         # print(mal_id)
#         image_path = "../data/images/" + str(mal_id) + ".jpg"
#         target_dir = Path("../data/images")
#         file_name = str(mal_id) + ".jpg"
#         image_path_true = Path(image_path)

#         if image_path_true.exists():
#             # print("Path found!")
#             img = Image.open(image_path)
#         else:
#             img = Image.new("RGB", (448, 448), (0, 0, 0))

#         img_tensor = transforms(img).unsqueeze(0)

#         with torch.no_grad():
#             embedding = model(img_tensor)
#             # print(embedding)

#         image_embeds.append(embedding)

#         count += 1

#         if count % 10 == 0:
#                 print(f"{count} images finished! Current MAL ID: {mal_id}")
#                 print(image_embeds)
#                 print(len(image_embeds))

#     image_embeds = torch.cat(image_embeds, dim=0)

#     return image_embeds

# image_embeddings = eva_tag(df4['mal_id'])

# print("Embedding Shape:", image_embeddings.shape)

In [ ]:
# FOR LOADING IMAGE EMBEDDINGS, CHANGE DEPENDING ON WHAT DATASET YOU'RE USING
# image_embeddings = np.load("../data/processed/image_embeddings.npy")


In [ ]:
# image_embeddings

Now we involve PCA.

In [ ]:
# pca_full = PCA().fit(image_embeddings)
# cumvar = np.cumsum(pca_full.explained_variance_ratio_)
# # find smallest n where cumvar[n] >= 0.90, e.g.
# n_90 = np.argmax(cumvar >= 0.90) + 1
# n_90

In [ ]:
# pca = PCA(n_components=n_90)
# compressed = pca.fit_transform(image_embeddings)
# compressed

In [ ]:
# target_dir = Path("../data/processed")
# file_path = target_dir / "image_embeddings.npy"
# np.save(file_path, compressed)

## Synopsis

We want to store synopses as a bunch of vectors via all-mpnet-base-v2.

In [ ]:
# sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
# embeddings = model.encode(sentences)
# print(embeddings)

In [ ]:
# embeddings = model.encode(
#     df4['synopsis'].tolist(),
#     batch_size=32,          # Adjust based on your VRAM
#     show_progress_bar=True,
#     convert_to_numpy=True
# )
# print(embeddings)
# print(embeddings.shape)

In [ ]:
# target_dir = Path("../data/processed")
# file_path = target_dir / "semantic_embeddings.npy"
# np.save(file_path, embeddings)

In [ ]:
df4.info()

## Multi-valued Features (REDO)

Five features can have multiple values: producers, genres, studios, demographics, and themes. Use Multi-Label Binarizer

Now we check producers and studios:

In [ ]:
all_studios = set(df4['studios'].explode().dropna())
all_producers = set(df4['producers'].explode().dropna())

overlap = all_studios & all_producers
print(f"Studios: {len(all_studios)}, Producers: {len(all_producers)}")
print(f"Overlapping names: {len(overlap)}")
print(f"% of producer entries that are also studios: {len(overlap)/len(all_producers):.1%}")

In [ ]:
def kfold_target_encode(df, col, target, n_splits=5):
    encoded = pd.Series(index=df.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(df):
        means = df.iloc[train_idx].explode(col).groupby(col)[target].mean()
        # for multi-valued rows, average across entries' encoded means
        encoded.iloc[val_idx] = df.iloc[val_idx][col].apply(
            lambda lst: pd.Series(lst).map(means).mean()
        )
    return encoded

df4['studio_enc'] = kfold_target_encode(df4, 'studios', 'score_z')
df4['producer_enc'] = kfold_target_encode(df4, 'producers', 'score_z')

print(df4[['studio_enc', 'producer_enc']].corr())

Reasonably low correlation. We can make a separate learned embedding from this. Now we bucket rare genres and themes, and perform one-hot-encoding.

In [ ]:
theme_counts = df4['themes'].explode().value_counts()
genre_counts = df4['genres'].explode().value_counts()

In [ ]:
print(theme_counts)
print(f"\nTotal unique themes: {len(theme_counts)}")
print(f"Themes appearing <5 times: {(theme_counts < 5).sum()}")
print(f"Themes appearing <10 times: {(theme_counts < 10).sum()}")

In [ ]:
threshold = 50
rare_themes = theme_counts[theme_counts < threshold].index.tolist()

def bucket_rare(theme_list, rare_set):
    return [t if t not in rare_set else "Other" for t in theme_list]

df4['themes_bucketed'] = df4['themes'].apply(lambda x: bucket_rare(x, set(rare_themes)))
print(sorted(rare_themes))

In [ ]:
print(genre_counts)
print(f"\nTotal unique genres: {len(genre_counts)}")
print(f"Genres appearing <5 times: {(genre_counts < 5).sum()}")
print(f"Genres appearing <10 times: {(genre_counts < 10).sum()}")

In [ ]:
threshold = 50
rare_genres = genre_counts[genre_counts < threshold].index.tolist()

def bucket_rare(genre_list, rare_set):
    return [t if t not in rare_set else "Other" for t in genre_list]

df4['genres_bucketed'] = df4['genres'].apply(lambda x: bucket_rare(x, set(rare_genres)))
print(sorted(rare_genres))

In [ ]:
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df4['genres_bucketed'])

genre_df = pd.DataFrame(
    genre_matrix,
    columns=[f"genre_{g}" for g in mlb.classes_], 
    index=df4.index
)

print(genre_df)

In [ ]:
theme_matrix = mlb.fit_transform(df4['themes_bucketed'])

theme_df = pd.DataFrame(
    theme_matrix,
    columns=[f"theme_{t}" for t in mlb.classes_],  
    index=df4.index
)

In [ ]:
demo_matrix = mlb.fit_transform(df['demographics'])

demo_df = pd.DataFrame(
    demo_matrix,
    columns=[f"demo_{d}" for d in mlb.classes_],  
    index=df4.index
)

In [ ]:
# Keep original categorical features for statistical analysis
df5 = df4.drop(columns=['synopsis', 'themes_bucketed', 'genres_bucketed', 'studio_enc', 'producer_enc'])
df5.columns = df5.columns.str.replace(' ', '_')
df5.info()

## Source Material

In [ ]:
df5.info()

In [ ]:
score_cols = ['manga_score_z', 'light_novel_score_z', 'novel_score_z', 'doujinshi_score_z', 'one_shot_score_z', 'manhwa_score_z', 'manhua_score_z']
member_cols = ['manga_members_z', 'light_novel_members_z', 'novel_members_z', 'doujinshi_members_z', 'one_shot_members_z', 'manhwa_members_z', 'manhua_members_z']

In [ ]:
df5['adaptation_score'] = df5[score_cols].apply(lambda row: np.nanmax(row.values) if row.notna().any() else np.nan, axis=1)
df5['adaptation_members'] = df5[member_cols].apply(lambda row: np.nanmax(row.values) if row.notna().any() else np.nan, axis=1)
print(df5['adaptation_score'])
print(df5['adaptation_members'])

In [ ]:
df5['has_adaptation_score'] = df5['adaptation_score'].notna().astype(int)
df5['has_adaptation_members'] = df5['adaptation_members'].notna().astype(int)
df6 = df5 # will fillna in notebook 5
df6['has_adaptation_score'].describe()

Now let's perform some extra EDA: I want to compare current strength vs. source material strength.

In [ ]:
adaptation_df = df6[df6['has_adaptation_score'] == 1]
adaptation_df.plot.scatter(x='adaptation_score', y='score_z', title='Source Material Score vs. Anime Score')
plt.show()

The zero line is because those without an adaptation score is imputed with a zero, but is supported by a "has_adaptation_score" boolean.

In [ ]:
adaptation_df = df6[df6['has_adaptation_members'] == 1]
adaptation_df.plot.scatter(x='adaptation_members', y='wc_z', title='Source Material Members vs. Anime WC')
plt.show()

In [ ]:
df7 = df6.drop(columns=['doujinshi_score_z', 'doujinshi_members_z', 'manhua_score_z', 'manhua_members_z', 'novel_score_z', 'novel_members_z',
                        'manhwa_score_z', 'manhwa_members_z', 'light_novel_score_z', 'light_novel_members_z', 'one_shot_score_z',
                        'one_shot_members_z', 'manga_score_z', 'manga_members_z'])
df7.info()

In [ ]:
df7['has_prequel_score'] = df7['prequel_score'].notna().astype(int)
df7['has_prequel_members'] = df7['prequel_members'].notna().astype(int)
df7['has_prequel_type'] = df7['prequel_type'].notna().astype(int)
df7['has_prequel_score'].describe()

In [ ]:
prequel_df = df7[df7['has_prequel_score'] == 1]
prequel_df.plot.scatter(x='prequel_score', y='score_z', title='Prequel Score vs. Anime Score')
plt.show()

In [ ]:
prequel_df = df7[df7['has_prequel_members'] == 1]
prequel_df['prequel_members'] = np.log1p(prequel_df['prequel_members'])
prequel_df.plot.scatter(x='prequel_members', y='wc_z', title='Prequel Members (Logged) vs. Anime WC')
plt.show()

In [ ]:
df8 = df7 # will fillna in notebook 5
df8.info()

I will remove the following columns:
* episodes
* source: adaptation columns take care of that
* sequel: prequel columns take care of that

In [ ]:
df9 = df8.drop(columns=['episodes', 'source'])
df9.info()

In [ ]:
df10 = pd.concat([df9, genre_df, theme_df, demo_df], axis=1)
df10.columns = df10.columns.str.replace(' ', '_')
df10.info()

In [ ]:
target_dir = Path("../data/processed")
file_path = target_dir / "anime_data_2.parquet"

df10.to_parquet(file_path, engine="pyarrow")